# 03 · Dating apps and going out (Q3 and Q4)
**Q3:** Are dating apps declining, or just changing?
**Q4:** What's happening to interest in going out, and has it recovered since COVID?

**Data** (produced by `src/clean.py`):
- `trends.csv`: Google Trends search interest in Ireland, monthly, Nov 2015 – Nov 2025. Each term is indexed to **its own 2016 average = 100**.
- `dating_apps.csv`: global paying users for Tinder, Hinge and Bumble App (company filings)

**How to read an index of 2016 = 100:** 200 means a term was searched twice as much as in 2016, and 50 means half as much.
It compares how each term has **changed**, not how popular the terms are relative to each other. Each term was exported separately, so Tinder at 50 and Hinge at 50 doesn't mean equal searches.

**Sections**
1. Dating apps: ten years of search interest
2. Searches in Ireland vs paying users worldwide
3. Going out: pub vs nightclub
4. Has anything recovered since COVID?
5. Summary table
6. Findings

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED = ROOT / "data" / "processed"
FIGURES = ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

trends = pd.read_csv(PROCESSED / "trends.csv")
apps = pd.read_csv(PROCESSED / "dating_apps.csv")
trends["date"] = pd.to_datetime(trends["month"])
trends["year"] = trends["month"].str[:4].astype(int)

plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "axes.titlesize": 13,
    "font.size": 11,
})
MUTED = "#a8a8a8"
COLOURS = {
    "tinder": "#da1e28", "hinge": "#0f62fe", "bumble": "#f1c21b",
    "pub": "#198038", "nightclub": "#8a3ffc",
}
COVID = (pd.Timestamp("2020-03-01"), pd.Timestamp("2022-01-31"))  # main closures to full reopening


def smooth(df, window=12):
    # 12-month rolling average removes seasonal spikes (Christmas, summer, January)
    # so the long-term trend is easier to see.
    return df.sort_values("date").assign(
        smoothed=lambda d: d["index_2016"].rolling(window, min_periods=window).mean()
    )


def shade_covid(ax):
    ax.axvspan(*COVID, color=MUTED, alpha=0.15, lw=0)
    ax.text(COVID[0], ax.get_ylim()[1], " COVID restrictions", va="top", fontsize=8, color="grey")


print("Terms:", ", ".join(trends.term.unique()))
print("Months:", trends.month.min(), "to", trends.month.max())

## 1. Dating apps: ten years of search interest
The faint lines are monthly values. The bold lines are 12-month rolling averages.

In [ ]:
app_terms = trends.loc[trends.group == "apps", "term"].unique()

fig, ax = plt.subplots()
for term in app_terms:
    d = smooth(trends[trends.term == term])
    c = COLOURS.get(term)
    ax.plot(d.date, d.index_2016, color=c, alpha=0.2, linewidth=1)
    ax.plot(d.date, d.smoothed, color=c, linewidth=2.5, label=term.title())
    last = d.dropna(subset=["smoothed"]).iloc[-1]
    ax.annotate(f"{last.smoothed:.0f}", xy=(last.date, last.smoothed), xytext=(6, 0),
                textcoords="offset points", va="center", color=c, fontweight="bold")

ax.axhline(100, color=MUTED, linestyle="--", linewidth=1)
shade_covid(ax)
ax.set_title("Irish searches for Tinder have halved. Hinge has taken off")
ax.set_ylabel("Search interest (2016 average = 100)")
ax.legend(frameon=False, loc="upper left")
fig.text(0.01, -0.02, "Source: Google Trends, Ireland. Bold = 12-month rolling average. "
         "Each term indexed to its own 2016 average.", fontsize=8, color="grey")
fig.tight_layout()
fig.savefig(FIGURES / "03_app_searches.png", dpi=150, bbox_inches="tight")
plt.show()

## 2. Searches in Ireland vs paying users worldwide
Do Irish searches move in the same direction as each company's global paying users?
Both are compared as **% change between the first and last year we have paying-user data for each app**.

In [ ]:
app_names = {"Tinder": "tinder", "Hinge": "hinge", "Bumble App": "bumble"}
yearly_search = trends.groupby(["term", "year"])["interest"].mean()

rows = []
for app, term in app_names.items():
    a = apps[apps.app == app].sort_values("year")
    y0, y1 = a.year.iloc[0], a.year.iloc[-1]
    rows.append({
        "app": app,
        "years": f"{y0}–{y1}",
        "paying_users_change_pct": round((a.paying_users_millions.iloc[-1] / a.paying_users_millions.iloc[0] - 1) * 100, 1),
        "irish_search_change_pct": round((yearly_search[(term, y1)] / yearly_search[(term, y0)] - 1) * 100, 1),
    })
comparison = pd.DataFrame(rows)
comparison["same_direction"] = (
    (comparison.paying_users_change_pct > 0) == (comparison.irish_search_change_pct > 0)
)
comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
x = range(len(comparison))
w = 0.38
b1 = ax.bar([i - w / 2 for i in x], comparison.paying_users_change_pct, w, color="#525252", label="Paying users (global)")
b2 = ax.bar([i + w / 2 for i in x], comparison.irish_search_change_pct, w,
            color=[COLOURS[app_names[a]] for a in comparison.app], label="Searches (Ireland)")
ax.bar_label(b1, fmt="%+.0f%%", padding=3, fontsize=9)
ax.bar_label(b2, fmt="%+.0f%%", padding=3, fontsize=9)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(list(x))
ax.set_xticklabels([f"{a}\n({y})" for a, y in zip(comparison.app, comparison.years)])
ax.set_ylabel("% change")
lo, hi = ax.get_ylim()
ax.set_ylim(lo - 8, hi + 8)   # room for the bar labels
ax.set_title("Paying users and Irish searches move in the same direction", pad=22)
ax.text(0.5, 1.02, "Grey = global paying users   ·   Colour = searches in Ireland",
        transform=ax.transAxes, ha="center", fontsize=9, color="grey")
fig.text(0.01, -0.03, "Sources: Match Group and Bumble Inc. filings (global); Google Trends (Ireland). "
         "Periods differ by app.", fontsize=8, color="grey")
fig.tight_layout()
fig.savefig(FIGURES / "03_searches_vs_payers.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Going out: pub vs nightclub

In [ ]:
out_terms = trends.loc[trends.group == "going_out", "term"].unique()

fig, ax = plt.subplots()
for term in out_terms:
    d = smooth(trends[trends.term == term])
    c = COLOURS.get(term)
    ax.plot(d.date, d.index_2016, color=c, alpha=0.2, linewidth=1)
    ax.plot(d.date, d.smoothed, color=c, linewidth=2.5, label=term.title())
    last = d.dropna(subset=["smoothed"]).iloc[-1]
    ax.annotate(f"{last.smoothed:.0f}", xy=(last.date, last.smoothed), xytext=(6, 0),
                textcoords="offset points", va="center", color=c, fontweight="bold")

ax.axhline(100, color=MUTED, linestyle="--", linewidth=1)
shade_covid(ax)
ax.set_title("Nightclub searches have halved since 2016 and never recovered from COVID")
ax.set_ylabel("Search interest (2016 average = 100)")
ax.legend(frameon=False, loc="lower left")
fig.text(0.01, -0.02, "Source: Google Trends, Ireland. Bold = 12-month rolling average.", fontsize=8, color="grey")
fig.tight_layout()
fig.savefig(FIGURES / "03_going_out_searches.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Has anything recovered since COVID?
Compare each term's **2025 average** with its **2019 average** (the last full year before COVID).
100% means it's back to pre-pandemic levels.

In [ ]:
recovery = (
    trends[trends.year.isin([2019, 2025])]
    .groupby(["term", "year"])["index_2016"].mean()
    .unstack()
)
recovery["pct_of_2019"] = (recovery[2025] / recovery[2019] * 100).round(0)
recovery = recovery.sort_values("pct_of_2019")

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh([t.title() for t in recovery.index], recovery.pct_of_2019,
               color=[COLOURS.get(t, MUTED) for t in recovery.index])
ax.bar_label(bars, fmt="%.0f%%", padding=3,
             bbox=dict(facecolor="white", edgecolor="none", pad=1))
ax.axvline(100, color="black", linestyle="--", linewidth=1, zorder=0)
ax.set_xlabel("2025 search interest as % of 2019  (dashed line = back to pre-COVID level)")
ax.set_title("Compared with before COVID: winners and losers")
fig.tight_layout()
fig.savefig(FIGURES / "03_recovery_since_covid.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Summary table

In [ ]:
by_year = trends.groupby(["group", "term", "year"])["index_2016"].mean().unstack()
summary = pd.DataFrame({
    "2016": by_year[2016].round(0),
    "2019": by_year[2019].round(0),
    "2025": by_year[2025].round(0),
})
summary["change_2016_to_2025"] = (summary["2025"] / summary["2016"] - 1).mul(100).round(0).astype(int).astype(str) + "%"
summary["change_2019_to_2025"] = (summary["2025"] / summary["2019"] - 1).mul(100).round(0).astype(int).astype(str) + "%"
summary

## 6. Findings
_Write these in your own words after looking at the charts. One sentence each._

**Q3: Dating apps**
1. **Tinder:** …
2. **Hinge and Bumble:** …
3. **Searches vs paying users:** …

**Q4: Going out**
4. **Nightclubs:** …
5. **Pubs:** …
6. **Since COVID:** …

**Caveats**
- **Search ≠ use.** People search for an app to download it, not every time they open it. Falling Tinder searches may partly mean people already have it installed.
- **Ambiguous words.** "Hinge" and "Bumble" are also ordinary English words, and "pub" includes searches for food, sport and directions. Hinge also started from a very low base in 2016, which exaggerates its growth.
- **Not age-specific.** Google Trends covers everyone in Ireland, not just young people.
- **Different geographies.** Paying users are **global**, searches are **Irish**. Matching directions are suggestive, not proof.
- **Q4 is incomplete.** These terms show what people are moving *away* from. To show what they're moving *towards*, add Google Trends exports for alternatives (e.g. *run club*, *five a side*, *speed dating*), add them to `TRENDS_FILES` in `clean.py`, and they'll appear in these charts automatically.